<a href="https://colab.research.google.com/github/fourmodern/2026_aidrugdiscovery/blob/main/Day06_LLM_Agent/t111_esm3_protein_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESM3 — 생성형 단백질 설계 (Day 2 추가 실습)

[EvolutionaryScale](https://www.evolutionaryscale.ai/)의 **ESM3**는 단백질의 **서열(sequence) · 구조(structure) · 기능(function)** 세 가지 트랙을 하나의 모델로 다루는 생성형 멀티모달 모델입니다.

> **예시 표적 (강의용): PDE5A** (phosphodiesterase type 5A; UniProt **O76074**, 875 aa) — PDE5 저해제(sildenafil, tadalafil)의 표적입니다. 아래 inpainting/구조예측 데모의 서열은 방법 시연을 위한 **일반 스캐폴드 서열**이며 실제 PDE5A 서열이 아닙니다. 실제 PDE5A를 설계 대상으로 쓰려면 UniProt O76074의 서열(또는 촉매 도메인 구간)을 불러와 마스킹하세요 (서열 확인 권장).

## t110(ESM-2)과 무엇이 다른가

| | ESM-2 (t110) | ESM3 (이 노트북) |
|---|---|---|
| 성격 | masked language model | 생성형 멀티모달 |
| 다루는 것 | 서열만 | 서열 + 구조 + 기능 |
| 사용 방식 | `EsmForMaskedLM` 로짓을 직접 스코어링 | `model.generate()` 반복 생성 |
| 구조 예측 | 불가 | 가능 (좌표 + pTM/pLDDT) |

t110의 `evo_prot_grad`는 **one-hot 입력에 대해 역전파 가능한 masked-LM 로짓**을 요구하므로 ESM3를 그 자리에 넣을 수 없습니다. 그래서 t110은 ESM-2를 유지하고, ESM3는 이렇게 별도 실습으로 둡니다.

## 이 노트북에서 하는 것

1. 서열 채우기(inpainting) — 마스크된 구간을 ESM3가 설계
2. 서열로부터 구조 예측 — 좌표와 신뢰도(pTM, pLDDT)
3. 예측 구조를 py3Dmol로 시각화

**GPU는 필수가 아닙니다.** CPU에서도 동작하며(이 노트북의 예시는 CPU에서 각 단계 약 13초), GPU가 있으면 더 빠릅니다.

## 0. 실행 환경 확인

**GPU는 필수가 아닙니다** (CPU에서도 각 생성 단계가 15초 내외). GPU가 있으면 더 빠릅니다.
Colab에서 GPU를 켜려면: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU

In [ ]:
# 실행 환경 확인 (Colab: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU)
import torch

print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[안내] CPU 런타임입니다. 실행은 되지만 느립니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

## 1. 설치

설치에 세 가지 주의사항이 있습니다.

1. **`--no-deps`가 필요합니다.** PyPI의 `esm` 패키지(EvolutionaryScale 배포, 현재 3.4.0)는
   `transformers>=4.57.6,<5.0.0` 과 `torch>=2.11,<2.12` 를 고정하고 있습니다. 그냥 설치하면
   Colab의 transformers/torch가 강제로 바뀌면서 런타임이 깨집니다.
   `--no-deps` 로 설치해도 이 노트북에서 쓰는 경로(ESM3 로드 / 서열·구조 생성 / PDB 출력)는
   최신 transformers 5.x + torch 2.13 조합에서 정상 동작하는 것을 확인했습니다.
2. **`torchtext`는 무시해도 됩니다.** 낡은 의존성 선언이며 패키지 코드에서 import하지 않습니다.
   (torchtext는 개발이 중단되었습니다)
3. **Python 버전.** `esm` 3.4.0 은 `requires-python >=3.12` 입니다. 현재 Colab(3.12)에서는 설치됩니다.
   Colab이 더 낮은 버전으로 돌아가면 `pip install esm==3.1.1` 처럼 낮은 버전을 지정하세요.

가중치 저장소는 원래 `EvolutionaryScale/esm3-sm-open-v1` 이었는데 조직명이 **`biohub`** 로 바뀌었습니다.
Hugging Face가 리다이렉트해 주므로 `ESM3.from_pretrained("esm3_sm_open_v1")` 코드는 그대로 동작합니다.
**라이선스 승인이나 HF 로그인은 필요 없습니다** (gated 아님).

In [ ]:
# ESM3 본체는 의존성 없이 설치 (transformers/torch 강제 다운그레이드 방지)
!pip install -q --no-deps esm

# esm이 런타임에 실제로 import하는 패키지들 중 Colab에 없는 것만 설치
!pip install -q zstd msgpack-numpy cloudpathlib tenacity pygtrie pydssp brotli
!pip install -q einops "biotite>=1.0.0" py3Dmol

import sys
print("Python:", sys.version.split()[0])
print("[확인] esm 은 Python >=3.12 를 요구합니다. 위 버전이 3.12 미만이면 설치가 실패합니다.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, GenerationConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

# 가중치는 Hugging Face에서 내려받습니다 (약 2.8GB, 라이선스 승인/로그인 불필요)
# 저장소: EvolutionaryScale/esm3-sm-open-v1
#   -> 조직명이 'biohub' 로 바뀌어 현재는 huggingface.co/biohub/esm3-sm-open-v1 로 리다이렉트됩니다.
#      esm 패키지가 알아서 따라가므로 코드는 그대로 두면 됩니다.
model = ESM3.from_pretrained("esm3_sm_open_v1", device=device)
print("ESM3 (esm3_sm_open_v1, 1.4B) 로드 완료 / 가중치 dtype:", next(model.parameters()).dtype)

# [중요] esm 3.x 체크포인트는 bfloat16 으로 저장되어 있는데,
# 모델 내부에서 pLDDT 임베딩이 float32 로 만들어져 아래 오류가 납니다.
#   RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16
# 모델 전체를 float32 로 올려 dtype을 통일합니다 (CPU 약 5.6GB RAM, T4 GPU에서도 충분).
model = model.to(torch.float32)
print("float32 로 캐스팅 완료:", next(model.parameters()).dtype)

## 2. 서열 채우기 (inpainting)

`_` 로 마스킹한 구간을 ESM3가 설계해 채웁니다. 실제 단백질 공학에서 루프 재설계나 링커 설계에 대응하는 작업입니다. (강의 맥락: 예시 표적 **PDE5A**의 루프/링커 재설계에 대응하는 작업으로 볼 수 있습니다. 아래 서열은 방법 시연용 일반 스캐폴드입니다.)

`GenerationConfig`의 주요 인자:
- `track="sequence"` — 서열 트랙을 생성
- `num_steps` — 반복 생성 횟수 (많으면 품질↑ 시간↑)
- `temperature` — 낮으면 보수적, 높으면 다양성↑

In [ ]:
# 앞부분 16잔기는 고정하고, 이어지는 12잔기를 ESM3가 설계하도록 마스킹
scaffold = "MKTVRQERLKSIVRIL"
tail = "ERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
masked_sequence = scaffold + "_" * 12 + tail

print(f"입력 길이: {len(masked_sequence)}  (마스크 12개)")
print(masked_sequence)

prompt = ESMProtein(sequence=masked_sequence)
designed = model.generate(
    prompt,
    GenerationConfig(track="sequence", num_steps=8, temperature=0.7),
)

print("\n설계된 서열:")
print(designed.sequence)
print("\nESM3가 채운 구간:", designed.sequence[len(scaffold):len(scaffold) + 12])
assert "_" not in designed.sequence, "마스크가 채워지지 않았습니다"

## 3. 서열로부터 구조 예측

같은 모델로 `track="structure"` 를 요청하면 3D 좌표를 생성합니다. 함께 나오는 신뢰도 지표는 AlphaFold와 같은 의미입니다.

- **pLDDT** (0~1): 잔기별 국소 신뢰도. 0.9 이상이면 매우 신뢰할 만합니다.
- **pTM** (0~1): 전체 접힘(fold) 정확도 추정. 0.5 이상이면 대체로 올바른 fold입니다.

In [ ]:
# 3단계에서 설계한 서열의 구조를 예측합니다
structure_prompt = ESMProtein(sequence=designed.sequence)
predicted = model.generate(
    structure_prompt,
    GenerationConfig(track="structure", num_steps=8),
)

print("좌표 shape:", tuple(predicted.coordinates.shape), "  (잔기 수, 원자 슬롯, xyz)")
print(f"pTM   : {float(predicted.ptm):.3f}")
print(f"pLDDT : {float(predicted.plddt.mean()):.3f} (평균)")

if float(predicted.ptm) < 0.5:
    print("\n주의: pTM이 낮습니다. num_steps를 늘리거나 서열을 다시 설계해 보세요.")

## 4. 예측 구조 시각화

`ESMProtein.to_pdb_string()` 으로 PDB 텍스트를 얻어 py3Dmol에 바로 올립니다.
(nglview는 Colab에서 렌더링되지 않아 이 워크숍의 다른 노트북과 동일하게 py3Dmol을 씁니다)

pLDDT로 색을 칠하면 어느 구간을 모델이 확신하는지 한눈에 보입니다.

In [ ]:
import py3Dmol

pdb_string = predicted.to_pdb_string()
print(f"PDB 텍스트 {len(pdb_string):,}자, ATOM 줄 {pdb_string.count(chr(10) + 'ATOM'):,}개")

# to_pdb_string() 은 B-factor 열에 pLDDT를 **0~100 스케일**로 씁니다
# (predicted.plddt 텐서는 0~1 스케일이라 단위가 다릅니다 - 색 범위를 0~100 으로 맞춰야 합니다)
b_values = [float(line[60:66]) for line in pdb_string.split("\n") if line.startswith("ATOM")]
print(f"B-factor(pLDDT) 범위: {min(b_values):.1f} ~ {max(b_values):.1f}")

view = py3Dmol.view(width=900, height=600)
view.addModel(pdb_string, "pdb")

# pLDDT 기반 색칠 (B-factor 열 0~100)
view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}}})

# ESM3가 새로 설계한 구간을 stick으로 강조
designed_region = list(range(len(scaffold) + 1, len(scaffold) + 13))  # PDB 잔기 번호는 1부터
view.addStyle({"resi": designed_region}, {"stick": {"radius": 0.15}})

view.zoomTo()
view.show()

print("색: 빨강(pLDDT 낮음) -> 파랑(pLDDT 높음) / stick: ESM3가 설계한 12잔기")

## 5. 정리와 한계

**확인한 것**
- 마스크 구간 서열 설계와 구조 예측이 동일한 모델에서 동작합니다.
- CPU에서도 각 단계가 15초 내외입니다 (1.4B open 모델, float32 기준).

**한계**
- `esm3_sm_open_v1`은 공개된 **소형(1.4B)** 모델입니다. 논문의 대형 모델(98B)은 [Forge API](https://forge.evolutionaryscale.ai/)를 통해서만 접근할 수 있습니다.
- 기능(function) 트랙은 InterPro 키워드 기반이며, 이 노트북에서는 다루지 않았습니다.
- 여기서 예측한 구조는 **모델의 추정**입니다. 실제 설계 검증에는 실험(발현·결합·열안정성)이 필요합니다.
- 체크포인트가 bfloat16이라 `model.to(torch.float32)` 캐스팅이 필요합니다(1절 참조). 캐스팅을 빼면
  `mat1 and mat2 must have the same dtype` 오류가 납니다.

**연결되는 실습 (`Day06_LLM_Agent/`)**
- `t110_esm2_peptide_optimization_tutorial` — ESM-2 기반 펩타이드 결합 최적화 (masked-LM 스코어링 방식)
- `t042_molt5` — 화학 언어모델(분자 ↔ 텍스트)
- `t050_simple-local-rag` — 문헌 기반 RAG